# 01 — Energy Landscape to Gating Rates

How molecular energy-landscape parameters (gating charge $z$, half-activation
$V_{half}$, symmetry $\delta$, prefactor $k_0$, rate caps) generate the
voltage-dependent transition rates $\alpha(V)$, $\beta(V)$, steady-state
$x_\infty(V)$, and time constant $\tau(V)$ for each HH gating particle.

A 2-state Eyring model reproduces the Boltzmann $x_\infty$ well. Rate caps
(Kramers limit) prevent $\tau$ from collapsing at depolarized voltages,
matching the saturating behaviour of classic HH rates.

In [1]:
import sys; sys.path.insert(0, '/workspace')
import os, warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.family'] = ['Liberation Sans','Arimo','DejaVu Sans']
matplotlib.rcParams['svg.fonttype'] = 'none'
FIG_DIR = '/mnt/results/hh_simulator/figures'
os.makedirs(FIG_DIR, exist_ok=True)
from hh_simulator import presets, viz
V = np.linspace(-100, 50, 601)
fits = {p: presets.fitted_landscape(p) for p in ('m','h','n')}
for p in ('m','h','n'): print(fits[p])

EnergyLandscape(name='m', k0=0.9971/ms, z=2.52, V_half=-40mV, delta=0.36)
EnergyLandscape(name='h', k0=0.05826/ms, z=-3.44, V_half=-62.3mV, delta=0.38)
EnergyLandscape(name='n', k0=0.08632/ms, z=1.37, V_half=-53.4mV, delta=0.74)


In [2]:
for p in ('m','h','n'):
    fig, axes = viz.plot_rates(p, V, savepath=f'{FIG_DIR}/01_rates_{p}.svg')
    plt.show()

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, p in zip(axes, ('m','h','n')):
    viz.plot_energy_landscape(fits[p], V, ax=ax)
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/01_energy_landscapes.svg', bbox_inches='tight')
fig.savefig(f'{FIG_DIR}/01_energy_landscapes.png', bbox_inches='tight', dpi=150)
plt.show()

In [4]:
print('particle | x_inf err | tau-shape err | tau_max | caps')
for p in ('m','h','n'):
    a,b = presets.CLASSIC_RATES[p]
    xinf_c = a(V)/(a(V)+b(V)); tau_c = 1.0/(a(V)+b(V))
    el = fits[p]
    ex = np.max(np.abs(el.x_inf(V)-xinf_c))
    et = np.max(np.abs(el.tau(V)/np.max(el.tau(V)) - tau_c/np.max(tau_c)))
    print(f'{p}        | {ex:.4f}    | {et:.4f}        | {np.max(tau_c):.3f}  | '
          f'a={el.alpha_cap:.2f} b={el.beta_cap:.2f}')

particle | x_inf err | tau-shape err | tau_max | caps
m        | 0.0162    | 0.1503        | 0.501  | a=9.00 b=inf
h        | 0.0109    | 0.0730        | 8.582  | a=inf b=1.00
n        | 0.0538    | 0.1187        | 5.792  | a=1.05 b=inf
